In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve, roc_curve

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import SelectKBest, mutual_info_classif

from sklearn.inspection import permutation_importance

from sklearn.utils.class_weight import compute_class_weight

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV, train_test_split

from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.feature_extraction.text import TfidfVectorizer
# import nltk
# from nltk.sentiment import SentimentIntensityAnalyzer

In [2]:
o_df = pd.read_csv("depression-classification-text-dataset.csv").dropna()

In [3]:
o_df

,text,label
0,"I'm feeling really down today, like everything...",1.0
1,"Just had a great workout, feeling fantastic an...",0.0
2,I can't seem to shake this feeling of emptines...,1.0
3,"Spent the day with friends, laughing and enjoy...",0.0
4,"I don't see the point in anything anymore, it ...",1.0
...,...,...
1583,Lost in a maze of endless possibilities.,1.0
1584,Seeking refuge in the sanctuary of solitude.,0.0
1585,Lost in the wilderness of my own mind.,1.0
1586,Finding strength in the face of adversity.,0.0


In [4]:
new_df = pd.DataFrame(o_df["text"])

In [5]:
new_df['char_count'] = new_df['text'].apply(len)
new_df['word_count'] = new_df['text'].apply(lambda x: len(x.split()))
new_df['avg_word_length'] = new_df['text'].apply(lambda x: sum(len(w) for w in x.split()) / len(x.split()))

In [6]:
new_df

,text,char_count,word_count,avg_word_length
0,"I'm feeling really down today, like everything...",64,10,5.500000
1,"Just had a great workout, feeling fantastic an...",63,11,4.818182
2,I can't seem to shake this feeling of emptines...,79,14,4.714286
3,"Spent the day with friends, laughing and enjoy...",71,11,5.545455
4,"I don't see the point in anything anymore, it ...",68,12,4.750000
...,...,...,...,...
1583,Lost in a maze of endless possibilities.,40,7,4.857143
1584,Seeking refuge in the sanctuary of solitude.,44,7,5.428571
1585,Lost in the wilderness of my own mind.,38,8,3.875000
1586,Finding strength in the face of adversity.,42,7,5.142857


In [7]:
df = pd.merge(o_df, new_df, on="text").drop_duplicates().reset_index(drop=True)

In [8]:
X = df.drop(columns=["text", "label"])

In [9]:
X

,char_count,word_count,avg_word_length
0,64,10,5.500000
1,63,11,4.818182
2,79,14,4.714286
3,71,11,5.545455
4,68,12,4.750000
...,...,...,...
229,40,7,4.857143
230,44,7,5.428571
231,38,8,3.875000
232,42,7,5.142857


In [10]:
# # Fit this on your full dataset
# vectorizer = TfidfVectorizer(max_features=1000)
# X_tfidf = vectorizer.fit_transform(X)  # where `list_of_texts` is your dataset

In [11]:
# tfidf_df = pd.DataFrame(
#     X_tfidf.toarray(),
#     columns=vectorizer.get_feature_names_out()
# )

In [12]:
# df_with_features = pd.concat([df.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)

In [13]:
# df_with_features

In [14]:
y = df["label"]

In [15]:
y

0      1.0
1      0.0
2      1.0
3      0.0
4      1.0
      ... 
229    1.0
230    0.0
231    1.0
232    0.0
233    1.0
Name: label, Length: 234, dtype: float64

In [16]:
# class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)

In [17]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)  # stratify=y

In [18]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [19]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    # ('feature_selection', SelectKBest(mutual_info_classif, k=10)),
    ('classifier', CalibratedClassifierCV(LinearSVC(max_iter=10000), cv=skf, method="sigmoid"))
])
# Grid search over LinearSVC parameters (use model__estimator__ to access nested parameters)
param_grid = {
    'classifier__estimator__C': [0.001, 0.01, 0.1, 1, 10, 100]
}

# GridSearchCV
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2,
)
grid.fit(X_train, y_train)

# Results
print("Best Parameters:", grid.best_params_)
print("Best Cross-Validation Score:", grid.best_score_)
# print("Test Accuracy:", grid.score(X_test, y_test))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best Parameters: {'classifier__estimator__C': 0.1}
Best Cross-Validation Score: 0.6971428571428572


C:\Users\corey\miniconda3\envs\dev\lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
C:\Users\corey\miniconda3\envs\dev\lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
C:\Users\corey\miniconda3\envs\dev\lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
C:\Users\corey\miniconda3\envs\dev\lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
C:\Users\corey\minic

In [20]:
result = permutation_importance(grid.best_estimator_, X_val, y_val, n_repeats=10)

In [21]:
result

{'importances_mean': array([0.03728814, 0.04745763, 0.00169492]),
 'importances_std': array([0.0319796 , 0.06148596, 0.03823903]),
 'importances': array([[ 0.05084746,  0.03389831,  0.03389831,  0.        , -0.03389831,
          0.05084746,  0.06779661,  0.08474576,  0.05084746,  0.03389831],
        [ 0.        , -0.05084746,  0.11864407,  0.11864407,  0.03389831,
          0.        ,  0.01694915,  0.06779661,  0.01694915,  0.15254237],
        [ 0.01694915, -0.01694915,  0.01694915,  0.01694915, -0.01694915,
         -0.08474576,  0.03389831, -0.01694915,  0.        ,  0.06779661]])}

In [22]:
# Predict
y_pred = grid.best_estimator_.predict(X_val)

In [23]:
y_pred

array([1., 0., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1., 1., 1.,
       0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 1.,
       0., 1., 0., 0., 1., 1., 1., 0., 1., 0., 0., 1., 1., 1., 1., 1., 1.,
       0., 0., 0., 0., 1., 1., 0., 0.])

In [24]:
y_probs = grid.best_estimator_.predict_proba(X_val)[:, 1]

In [25]:
y_probs

array([0.57421232, 0.26905863, 0.36899114, 0.29031056, 0.62595863,
       0.26905863, 0.7370388 , 0.75689586, 0.33357613, 0.41750395,
       0.33271857, 0.17763224, 0.7419131 , 0.49165053, 0.58728783,
       0.53993961, 0.67123606, 0.35933851, 0.28230847, 0.60058008,
       0.44852779, 0.46395194, 0.28502346, 0.43613737, 0.26396026,
       0.35933851, 0.67123606, 0.58728783, 0.35933851, 0.36365192,
       0.41008585, 0.33667278, 0.49165053, 0.51946414, 0.38767171,
       0.6019153 , 0.26396026, 0.21093429, 0.85344922, 0.70115738,
       0.64150054, 0.46966322, 0.52913029, 0.24466717, 0.28502346,
       0.78211642, 0.698433  , 0.66035181, 0.61487957, 0.62595863,
       0.54458166, 0.31046866, 0.14071694, 0.41185343, 0.10770142,
       0.66329597, 0.84574772, 0.33271857, 0.48035319])

In [26]:
thresholds = np.linspace(0, 1, 100)
best_threshold = 0.5
best_f1 = 0

# Store metrics
results = {
    'Threshold': [],
    'Precision': [],
    'Recall': [],
    'F1 Score': []
}

f1_scores = []

for threshold in thresholds:
    y_pred = (y_probs >= threshold).astype(int)
    f1 = f1_score(y_val, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold
    f1_scores.append(f1)

    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    results['Threshold'].append(threshold)
    results['Precision'].append(precision)
    results['Recall'].append(recall)
    results['F1 Score'].append(f1)

print(f"Best threshold: {best_threshold:.2f}")
print(f"Best F1 score: {best_f1:.4f}")

Best threshold: 0.14
Best F1 score: 0.6588


In [27]:
df_results = pd.DataFrame(results)
print(df_results.sort_values('F1 Score', ascending=False).head())

    Threshold  Precision    Recall  F1 Score
17   0.171717   0.491228  1.000000  0.658824
16   0.161616   0.491228  1.000000  0.658824
15   0.151515   0.491228  1.000000  0.658824
14   0.141414   0.491228  1.000000  0.658824
25   0.252525   0.500000  0.964286  0.658537


In [28]:
precisions, recalls, thresholds = precision_recall_curve(y_val, y_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)

print(f"Best threshold: {thresholds[best_idx]:.2f}")
print(f"Best F1 score: {f1_scores[best_idx]:.4f}")

Best threshold: 0.27
Best F1 score: 0.6750


In [29]:
# # Plot
# plt.figure(figsize=(8, 5))
# plt.plot(thresholds, f1_scores, label='F1 Score')
# plt.xlabel('Threshold')
# plt.ylabel('F1 Score')
# plt.title('F1 Score vs. Classification Threshold')
# plt.grid(True)
# plt.legend()
# plt.show()

In [30]:
# plt.figure(figsize=(10, 6))
# plt.plot(df_results['Threshold'], df_results['Precision'], label='Precision')
# plt.plot(df_results['Threshold'], df_results['Recall'], label='Recall')
# plt.plot(df_results['Threshold'], df_results['F1 Score'], label='F1 Score')
# plt.xlabel('Threshold')
# plt.ylabel('Metric Score')
# plt.title('Precision, Recall, F1 vs. Threshold')
# plt.grid(True)
# plt.legend()
# plt.tight_layout()
# plt.show()